# State and Matrix Oracle Sketching

This notebook is a continuation of [Quantum Oracle Sketching](qauntum_oracle_sketching.ipynb). It extends the per-sample phase-rotation primitive built there from a Boolean target function to (i) real-valued vectors — yielding a state-preparation routine — and (ii) sparse matrices — yielding a block encoding. Familiarity with the Boolean construction and its $M = \Theta(N/\epsilon)$ sample-complexity argument is assumed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from oracle_sketching import (
    state_sketch,
    state_sketch_circuit,
    stream_sketch,
)

from classiq import *

np.random.seed(7)

NUM_QUBITS = 3
N = 2**NUM_QUBITS

# A random Boolean target function f: [N] -> {0, 1} (used by the streaming demo below).
F_TABLE = np.random.randint(0, 2, size=N)
print(f"f(x) for x = 0,...,{N-1}:  {F_TABLE.tolist()}")

## Overview

The Boolean phase oracle is the cleanest setting in which to see the oracle-sketching algorithm at work, but the same primitive — an incremental, basis-state-targeted phase rotation — extends to all of the standard quantum-input formats. The constructions sketched in Sec. D.4–D.6 of [[1]](#ref1) layer two ideas on top of what was built in the [Boolean notebook](qauntum_oracle_sketching.ipynb):

1. **Per-sample real-valued angles.** Replacing the Boolean $f(x_t) \in \{0,1\}$ with a continuous value $v_{x_t} \in [-1, 1]$ and rescaling $\tau$ accordingly turns the sketch into a phase oracle for any real-valued function — without changing the gate structure.
2. **Hadamard-test conversion plus oblivious amplitude amplification.** A single ancilla qubit converts the phase oracle into a block encoding of a diagonal matrix; *oblivious amplitude amplification* (OAA) then raises that block to unit amplitude, yielding a deterministic state-preparation unitary.

Below we walk through both extensions and close with the streaming runtime view that motivated the algorithm in the first place.

### State sketching for real-valued vectors

**Goal.** Given a real unit vector $\vec{v}\in\mathbb{R}^N$ with $|v_x| \le 1$, prepare $|\vec{v}\rangle = \sum_x v_x|x\rangle$ from classical samples $(x_t,\,v_{x_t})$ with $x_t \sim \mathrm{Unif}([N])$.

**Sketched phase oracle.** Generalising the per-sample gate to admit a real-valued angle,

$$
V_t \;=\; \exp\!\bigl(i\,\tau\, v_{x_t}\, |x_t\rangle\!\langle x_t| / M\bigr),
$$

the same product-and-concentration argument gives

$$
V \;\longrightarrow\; \sum_x e^{i \tau\, p(x)\, v_x}\,|x\rangle\!\langle x|.
$$

Choosing $\tau = \theta N$ with a small *amplitude angle* $\theta \ll 1$ produces the phase oracle $V|x\rangle = e^{i\theta v_x}\,|x\rangle$.

**Hadamard-test conversion.** A single ancilla qubit converts the phase into an amplitude. Starting from $|0\rangle_a \otimes |+\rangle^{\otimes n}$ and applying $H_a \cdot \mathrm{C\text{-}}V \cdot H_a$,

$$
|0\rangle_a |+\rangle^{\otimes n} \;\longmapsto\; \frac{1}{\sqrt{N}}\sum_x \Bigl(\cos\!\tfrac{\theta v_x}{2}\,|0\rangle_a + i\,\sin\!\tfrac{\theta v_x}{2}\,|1\rangle_a\Bigr) |x\rangle.
$$

Post-selecting on $|1\rangle_a$ leaves the state $\propto \sum_x \sin(\theta v_x/2)\,|x\rangle \approx (\theta/2)\,|\vec v\rangle$ for $\theta \to 0$. The post-selection success probability is $p_\text{succ} = \Theta(\theta^2)$.

**Oblivious amplitude amplification (OAA).** Applying OAA $\mathcal{O}(1/\theta)$ times raises the post-selected branch to unit amplitude, yielding a *deterministic* state-preparation unitary $U_v$ with $U_v|0\rangle = |\vec v\rangle$ at the cost of $\mathcal{O}(1/\theta)$ additional sketch invocations. We omit the OAA inner loop in the demo below — it is a fixed sequence of reflections, mechanical to implement — and instead verify the *post-selected* amplitude vector against the target.

In [ ]:
# Toy target: a random real unit vector
NUM_QUBITS_V = 3
N_V = 2**NUM_QUBITS_V
np.random.seed(11)
v_target = np.random.randn(N_V)
v_target /= np.linalg.norm(v_target)

# Sketch
amp_angle = np.pi / 16
M_v = 5000
x_samples_v = np.random.randint(0, N_V, size=M_v)
amps = state_sketch(v_target, x_samples_v, amp_angle)

# In the small-θ limit, amps ≈ (θ/2) v_x / sqrt(N).
# Recover the estimated state and check its overlap with the target.
v_est = amps / (amp_angle / 2) * np.sqrt(N_V)
v_est /= np.linalg.norm(v_est)
print(f"target |v⟩ :  {np.round(v_target, 3)}")
print(f"v_est      :  {np.round(v_est, 3)}")
print(f"|⟨v_est | v⟩| = {abs(v_est @ v_target):.4f}    (1 = perfect)")

In [ ]:
M_values_v = np.unique(np.logspace(2, 4.5, 12, dtype=int))
n_trials_v = 100

infidelity = np.empty(M_values_v.size)
for k, M_k in enumerate(M_values_v):
    fids = []
    for _ in range(n_trials_v):
        xs = np.random.randint(0, N_V, size=int(M_k))
        a = state_sketch(v_target, xs, amp_angle)
        v_est_k = a / (amp_angle / 2) * np.sqrt(N_V)
        v_est_k /= np.linalg.norm(v_est_k) + 1e-15
        fids.append(abs(v_est_k @ v_target))
    infidelity[k] = 1 - np.mean(fids)

c_fit = float(np.median(infidelity * M_values_v / N_V))

fig, ax = plt.subplots(figsize=(5, 4))
ax.loglog(
    M_values_v, infidelity, "o-", label=r"$1 - |\langle v_\mathrm{est}\, |\, v\rangle|$"
)
ax.loglog(
    M_values_v, c_fit * N_V / M_values_v, "--", label=rf"${c_fit:.2f}\,N/M$ guide"
)
ax.set_xlabel("Number of samples $M$")
ax.set_ylabel("Infidelity")
ax.set_title(f"State sketching, $N={N_V}$, $\\theta=\\pi/16$, {n_trials_v} trials")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

**Classiq circuit for state sketching.** The numpy reference above operates directly on diagonal phases. Below we build the same construction as a Classiq `@qfunc` so the post-selected branch becomes an actual quantum state-preparation routine. The circuit has three pieces:

1. `real_valued_oracle_sketch` — a real-valued generalisation of `quantum_oracle_sketch` where each sample $t$ carries its *own* rotation angle (the only way a single fixed $\theta$ is no longer enough is precisely the move from Boolean $f$ to real-valued $\vec v$).
2. `state_sketch_circuit` — the Hadamard-test wrapper: `H(aux); control(aux == 1, real-valued sketch); H(aux)`. Conditioning the entire sketched phase oracle on `aux == 1` is what converts the diagonal phase $e^{i\theta v_x}$ into amplitudes $\sin(\theta v_x/2)$ on the $|1\rangle_a$ branch.
3. `state_sketch_main` — bakes $M = 200$ classical samples at $\theta = \pi/16$ into the circuit (the "compile-time" view from the Boolean section), allocates the registers, applies a Hadamard transform to set up the uniform superposition, and then runs the sketch. Post-selecting `aux == 1` recovers the unnormalised target amplitudes $\sin(\theta v_x/2)\,|x\rangle \approx (\theta/2)\,|\vec v\rangle$.

Synthesis is rendered for inspection — full QSVT-style amplitude amplification (which would lift the post-selected branch to unit amplitude) lives in the LS-SVM / PCA notebooks of this series.

In [ ]:
# Bake angles_per_sample[t] = θ N v_{x_t} / M so the inner product τ m_x v_x
# converges to θ v_x.
M_state_demo = 200
samples_state_demo = np.random.randint(0, N_V, size=M_state_demo)
angles_demo = (amp_angle * N_V / M_state_demo * v_target[samples_state_demo]).tolist()


# Classiq requires the entry-point qfunc to be named `main`.
@qfunc
def main(aux: Output[QBit], qvar: Output[QNum]) -> None:
    allocate(1, aux)
    allocate(NUM_QUBITS_V, qvar)
    hadamard_transform(qvar)
    state_sketch_circuit(angles_demo, samples_state_demo.tolist(), aux, qvar)


qprog_state = synthesize(main)
show(qprog_state)

### Block encodings of sparse matrices

State sketching extracts a *vector* from classical data. The next step — extracting a *matrix* — relies on the same primitive but with two parallel oracles: a *matrix-element* oracle that returns $A_{ij}$ given the index pair $(i, j)$, and a *matrix-index* oracle that returns the column index of the $k$-th non-zero entry in row $i$. Both are sketched using the per-sample phase rotation we have already built, and both are then combined into a *block encoding* $U_A$ on $a + n$ qubits satisfying

$$
\bigl(\langle 0|^{\otimes a} \otimes I_n\bigr)\, U_A\, \bigl(|0\rangle^{\otimes a} \otimes I_n\bigr) \;=\; A / \alpha,
$$

for some normalisation $\alpha = \mathcal{O}(\|A\|_\text{max} \cdot s)$ with $s$ the row sparsity. The construction needs $\widetilde{\mathcal{O}}(N)$ classical samples per query, matching the $M = \Theta(N/\epsilon)$ scaling we proved for the Boolean case.

Once $A$ is block-encoded, the [quantum singular-value transformation](https://arxiv.org/abs/1806.01838) [[4]](#ref4) yields a polynomial of $A$ — including $A^{-1}$ for HHL-style linear-system solving and the spectral projector for PCA. This is the path to the LS-SVM and PCA tasks of Sec. F of [[1]](#ref1) and is the subject of a future notebook in this series; the QSVT machinery is substantial enough that it deserves a dedicated exposition rather than a few cells appended here.

### Compile-time vs streaming circuits

The Classiq circuit synthesised earlier bakes all $M$ samples into a single program — convenient for benchmarking but not the runtime model the algorithm was designed for. In a true streaming deployment, samples arrive one at a time; the quantum learner applies the corresponding gate immediately and discards the sample, never holding more than its current quantum state and an $\mathcal{O}(\log N)$-size register of running statistics.

The numerical content is identical — every $V_t$ is diagonal so gate ordering is irrelevant — but the runtime profile is very different: the per-sample cost is constant, and the wall-clock time scales with the data feed rather than with a one-shot compilation of the full $M$-gate circuit. In Classiq this is naturally expressed as an `ExecutionSession` that applies one fresh `apply_basis_phase` gate per arriving sample on top of the persisted quantum state.

Below we simulate the streaming view in numpy: we accumulate empirical frequencies sample by sample and snapshot the resulting unitary $V$ at increasing sample counts. The error to the ideal oracle decays smoothly as more data arrives — without ever materialising the full sample list in memory.

In [ ]:
checkpoints = [100, 250, 500, 1000, 2500, 5000, 10000]
rng = np.random.default_rng(0)
snaps = stream_sketch(F_TABLE, max(checkpoints), checkpoints, rng=rng)

print(f"{'samples':>10s}  {'||V - O||_2':>12s}")
for t, err in snaps:
    print(f"{t:>10d}  {err:>12.3e}")

## What's next

- **End-to-end LS-SVM on classical data.** Combine the matrix block encoding above with QSVT-based matrix inversion to solve a least-squares support-vector machine using only streaming classical samples (Theorem 3 of [[1]](#ref1)).
- **Dimension reduction (PCA).** Use the same block encoding to project test points onto the top principal direction of a sparse, gapped data matrix (Theorem 5 of [[1]](#ref1)).
- **Interferometric classical shadows for readout.** Compress the resulting quantum model into a compact classical predictor using the sign-preserving shadow protocol (Theorem F.16 of [[1]](#ref1), built on the Huang–Kueng–Preskill construction [[2]](#ref2)) — the final stage of the three-stage pipeline introduced earlier in this notebook.

These will be the subjects of follow-up notebooks in this series.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. Exponential quantum advantage in processing massive classical data. [arXiv:2604.07639 (2026)](https://arxiv.org/abs/2604.07639)

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). https://doi.org/10.1038/s41567-020-0932-7. [arXiv](https://arxiv.org/abs/2002.08953)

<a id='ref3'></a>
[[3]](#ref3) Giovannetti, V., Lloyd, S., and Maccone, L. *Quantum random access memory.* Physical Review Letters 100, 160501 (2008). https://doi.org/10.1103/PhysRevLett.100.160501. [arXiv](https://arxiv.org/abs/0708.1879)

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366. [arXiv](https://arxiv.org/abs/1806.01838).

<a id='ref5'></a>
[[5]](#ref5) Harrow, A. W., Hassidim, A., and Lloyd, S. *Quantum algorithm for linear systems of equations.* Physical Review Letters 103, 150502 (2009). https://doi.org/10.1103/PhysRevLett.103.150502. [arXiv](https://arxiv.org/abs/0811.3171).